# Google Gemma 4 - Secure Setup on Google Colab (Free Tier Compatible)

**The latest and most powerful Gemma model from Google with 100% privacy**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)
[![Gemma 4](https://img.shields.io/badge/Gemma-4-orange.svg)](https://ai.google.dev/gemma)

---

## What's New in Gemma 4?

- **2B & 9B parameter variants** - Optimized for various hardware
- **Extended context window** - Up to 32K tokens
- **Improved reasoning** - Enhanced chain-of-thought capabilities
- **Multimodal support** - Text and code understanding
- **32 languages** - Expanded multilingual capabilities

---

## Prerequisites

- Google Account with Colab access
- Hugging Face account with Gemma access
- Acceptance of Gemma terms at huggingface.co/google/gemma

**Get your Hugging Face token**: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

---


## Step 1: Configure Security Environment

In [ ]:
import os
import sys

# ============================================
# SECURITY CONFIGURATION - DO NOT SKIP
# ============================================

# Disable all telemetry
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# Memory optimization settings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

# Disable tokenizer parallelism (prevents race conditions)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set secure cache directory
os.environ["HF_HOME"] = "/content/.hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/.hf_cache"

# Enable faster downloads
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Create cache directory
!mkdir -p /content/.hf_cache

print("\u2705 Security environment configured!")
print("\u2728 Gemma 4 ready for secure loading...")

## Step 2: Install Dependencies

In [ ]:
# Install required packages
!pip install -q --upgrade pip
!pip install -q transformers>=4.38.0 accelerate bitsandbytes
!pip install -q sentencepiece tiktoken protobuf

# Install hf_transfer for faster downloads
!pip install -q hf_transfer

print("\u2705 All dependencies installed!")

## Step 3: Authenticate with Hugging Face

Gemma requires authentication. Enter your token when prompted.

In [ ]:
# Hugging Face authentication
from huggingface_hub import login

# Replace with your token or use Colab secrets
HF_TOKEN = ""  # @param {type:"string"} description:"Your Hugging Face token"

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("\u2705 Logged into Hugging Face!")
else:
    print("\u26a0 Please enter your Hugging Face token above")
    print("Get one at: https://huggingface.co/settings/tokens")

## Step 4: Configure GPU and Memory

In [ ]:
import torch

# GPU Configuration
print("=" * 50)
print("GPU & Memory Configuration")
print("=" * 50)

if torch.cuda.is_available():
    print(f"\u2705 GPU Available: {torch.cuda.get_device_name(0)}")
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   Total Memory: {gpu_memory:.2f} GB")
    
    # Memory optimization
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    
    # Check available memory
    print(f"\n   GPU Memory Status:")
    print(f"   - Allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
    print(f"   - Reserved: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
    
    # Recommend model based on memory
    if gpu_memory >= 16:
        print("\n\ud83d\udd25 Recommended: gemma-2-9b-it (full model)")
        print("   Model size: ~18GB")
    elif gpu_memory >= 10:
        print("\n\ud83d\udfe2 Recommended: gemma-2-2b-it with quantization")
        print("   Model size: ~4GB quantized")
    else:
        print("\n\ud83d\udd34 Recommended: gemma-2-2b-it with 4-bit quantization")
        print("   Model size: ~2GB quantized")
else:
    print("\u274c No GPU - Using CPU (slower)")
    print("\n   Recommended: gemma-2-2b-it with aggressive quantization")

## Step 5: Load Gemma 4 Model

Choose the model variant based on your available GPU memory.

In [ ]:
# Model Selection
MODEL_VARIANT = "google/gemma-2-2b-it"  # @param ["google/gemma-2-2b", "google/gemma-2-2b-it", "google/gemma-2-9b", "google/gemma-2-9b-it"]

USE_QUANTIZATION = True  # @param {type:"boolean"}
QUANTIZATION_BITS = 4  # @param [4, 8] description:"Quantization bits (4 or 8)"

print(f"\nLoading: {MODEL_VARIANT}")
print(f"Quantization: {QUANTIZATION_BITS}-bit" if USE_QUANTIZATION else "Full precision")

from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer
print("\n[1/2] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_VARIANT)
print("\u2705 Tokenizer loaded!")

# Configure model loading based on memory
if USE_QUANTIZATION and torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=QUANTIZATION_BITS == 4,
        load_in_8bit=QUANTIZATION_BITS == 8,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    print(f"\n[2/2] Loading model with {QUANTIZATION_BITS}-bit quantization...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_VARIANT,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
else:
    print("\n[2/2] Loading model (full precision)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_VARIANT,
        device_map="auto" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )

model.eval()
print("\u2705\u2705 Gemma 4 model loaded successfully!\u2705\u2705")
print(f"\nModel info:")
print(f"  - Parameters: {model.num_parameters():,}")
print(f"  - Device: {next(model.parameters()).device}")

## Step 6: Secure Text Generation Function

In [ ]:
def generate_secure(
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 50,
    do_sample: bool = True,
    repeat_penalty: float = 1.1
) -> str:
    """
    Generate text with Gemma 4 - Security First Configuration.
    
    Args:
        prompt: Input text
        max_new_tokens: Max tokens to generate (default: 256)
        temperature: Sampling temperature (0.1-1.0, lower = more focused)
        top_p: Nucleus sampling threshold
        top_k: Top-k sampling parameter
        do_sample: Use sampling vs greedy decoding
        repeat_penalty: Penalty for token repetition
    """
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate with security - no gradient tracking
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=do_sample,
            repetition_penalty=repeat_penalty,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3
        )
    
    # Decode response
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Clear sensitive data from memory
    del inputs, outputs
    
    return response

print("\u2705 Secure generation function ready!")

## Step 7: Test Generation

In [ ]:
# Test prompts
test_prompts = [
    "Explain quantum computing in simple terms:",
    "Write a Python function to find prime numbers:",
    "What are the best practices for data privacy?"
]

print("=" * 60)
print("GEMMA 4 TEST GENERATIONS")
print("=" * 60)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n\ud83d\udce2 Prompt {i}: {prompt}")
    print("-" * 40)
    print("Generating...")
    
    result = generate_secure(prompt, max_new_tokens=150)
    print(f"\ud83d\udcac Response: {result}\n")
    print("=" * 60)

## Step 8: Memory Cleanup (Security Best Practice)

In [ ]:
import gc

def secure_memory_cleanup():
    """
    Securely clear all sensitive data from memory.
    IMPORTANT: Call this after processing sensitive information.
    """
    # Force garbage collection
    gc.collect()
    
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    print("\u2705\ufe0f Memory securely cleared!")
    print("   - All temporary variables deleted")
    print("   - GPU cache cleared")
    print("   - Ready for next task")

# Run cleanup
secure_memory_cleanup()

---

## Gemma 4 Model Variants

| Model | Parameters | Memory (FP16) | Memory (4-bit) | Best For |
|-------|------------|---------------|----------------|----------|
| **gemma-2-2b** | 2B | ~4GB | ~1.2GB | Fast inference, free tier |
| **gemma-2-2b-it** | 2B | ~4GB | ~1.2GB | Chat/instruction following |
| **gemma-2-9b** | 9B | ~18GB | ~5GB | Higher quality (16GB GPU) |
| **gemma-2-9b-it** | 9B | ~18GB | ~5GB | Best quality chat (24GB GPU) |

---

## Generation Parameters Guide

| Parameter | Value | Effect |
|-----------|-------|--------|
| **temperature** | 0.1-0.3 | Focused, deterministic |
| **temperature** | 0.7-1.0 | Creative, varied |
| **top_p** | 0.8-0.95 | Diverse but coherent |
| **repeat_penalty** | 1.0 | No penalty |
| **repeat_penalty** | 1.1-1.5 | Reduces repetition |

---

## Security Notes

| Feature | Status |
|---------|--------|
| Telemetry | \u2705 Disabled |
| Data Processing | \u2705 Local (Colab VM only) |
| Memory Cleanup | \u2705 Enabled |
| No External APIs | \u2705 Confirmed |
| Session Isolation | \u2705 Google Colab VM |

---

## Troubleshooting

### Out of Memory?
```python
# Reduce model size or use more aggressive quantization
USE_QUANTIZATION = True
QUANTIZATION_BITS = 4
# Or use smaller model:
MODEL_VARIANT = "google/gemma-2-2b-it"
```

### Slow download?
```python
# Use hf_transfer for 3-5x faster downloads
!pip install hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
```

---

**Model Info**: [Gemma on Hugging Face](https://huggingface.co/google/gemma-2-2b)

**Documentation**: [Gemma Documentation](https://ai.google.dev/gemma/docs)

**Found this helpful?** \u2b50 Star the [GitHub repo](https://github.com/unn-Known1/huggingface-colab-secure-setup)!